In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e2/sample_submission.csv
/kaggle/input/playground-series-s6e2/train.csv
/kaggle/input/playground-series-s6e2/test.csv
/kaggle/input/playground-series-s6e2-train-combined/train_combined.csv


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer, TargetEncoder, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

In [3]:
dataset_path = '/kaggle/input/playground-series-s6e2'

In [4]:
train = pd.read_csv(f'{dataset_path}/train.csv')
train.head()

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence


In [5]:
train_df = train.copy()
train_df['Heart Disease'] = (train_df['Heart Disease'] == 'Presence').astype(int)

X = train_df.drop(['Heart Disease'], axis=1)
y = train_df['Heart Disease']

In [6]:
def feature_eng(X: pd.DataFrame):
    X = X.copy()
    if 'id' in X.columns:
        X = X.drop(['id'], axis=1)
    
    X = X.drop(['FBS over 120'], axis=1)
    
    return X

In [7]:
cat_cols = ['Chest pain type', 'EKG results', 'Slope of ST', 'Thallium']

encoder_transformer = ColumnTransformer(
    transformers=[
        ('target_enc', TargetEncoder(random_state=42), cat_cols),
    ],
    remainder='passthrough'
)

preprocessor = Pipeline([
    ('feature_eng', FunctionTransformer(feature_eng)),
    ('encoder', encoder_transformer),
    ('scaler', StandardScaler())
])

In [8]:
xgb_params = {'n_estimators': 962, 'learning_rate': 0.037395158129910566, 'max_depth': 5, 'min_child_weight': 5, 'subsample': 0.9729126220493468, 'colsample_bytree': 0.5158352066892495, 'gamma': 0.15744297565813703}
cat_params = {'iterations': 838, 'learning_rate': 0.09981859646361525, 'depth': 3, 'l2_leaf_reg': 3, 'subsample': 0.9291146360119862, 'colsample_bylevel': 0.5159046186312849, 'silent': True}
lgbm_params = {'n_estimators': 838, 'learning_rate': 0.09981859646361525, 'num_leaves': 151, 'max_depth': 3, 'min_child_samples': 86, 'subsample': 0.9291146360119862, 'colsample_bytree': 0.5159046186312849, 'verbose': -1}

In [9]:
estimators = [
    ('xgb', XGBClassifier(**xgb_params, random_state=42)),
    ('cat', CatBoostClassifier(**cat_params, random_state=42)),
    ('lgbm', LGBMClassifier(**lgbm_params, random_state=42))
]

stacking_model = Pipeline([
    ('preprocessor', preprocessor),
    ('stacking', StackingClassifier(
        estimators=estimators,
        final_estimator=LogisticRegression(),
        stack_method='predict_proba',
        cv=5,
        n_jobs=-1
    ))
])

In [10]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
stacking_model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/co

Pipeline(steps=[('preprocessor',
                 Pipeline(steps=[('feature_eng',
                                  FunctionTransformer(func=<function feature_eng at 0x78f6785bcae0>)),
                                 ('encoder',
                                  ColumnTransformer(remainder='passthrough',
                                                    transformers=[('target_enc',
                                                                   TargetEncoder(random_state=42),
                                                                   ['Chest '
                                                                    'pain type',
                                                                    'EKG '
                                                                    'results',
                                                                    'Slope of '
                                                                    'ST',
                                                                    'Thallium'])])),
                                 ('scaler', StandardScaler())])),
                ('stacking',
                 Sta...
                                                 <catboost.core.CatBoostClassifier object at 0x78f67957fbc0>),
                                                ('lgbm',
                                                 LGBMClassifier(colsample_bytree=0.5159046186312849,
                                                                learning_rate=0.09981859646361525,
                                                                max_depth=3,
                                                                min_child_samples=86,
                                                                n_estimators=838,
                                                                num_leaves=151,
                                                                random_state=42,
                                                                subsample=0.9291146360119862,
                                                                verbose=-1))],
                                    final_estimator=LogisticRegression(),
                                    n_jobs=-1, stack_method='predict_proba'))])

In [11]:
y_pred_proba = stacking_model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, y_pred_proba)
print(f'StackingClassifier ROC-AUC: {auc:.4f}')

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


StackingClassifier ROC-AUC: 0.9563


In [12]:
print("Retraining on full dataset...")
stacking_model.fit(X, y)

Retraining on full dataset...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/co

Pipeline(steps=[('preprocessor',
                 Pipeline(steps=[('feature_eng',
                                  FunctionTransformer(func=<function feature_eng at 0x78f6785bcae0>)),
                                 ('encoder',
                                  ColumnTransformer(remainder='passthrough',
                                                    transformers=[('target_enc',
                                                                   TargetEncoder(random_state=42),
                                                                   ['Chest '
                                                                    'pain type',
                                                                    'EKG '
                                                                    'results',
                                                                    'Slope of '
                                                                    'ST',
                                                                    'Thallium'])])),
                                 ('scaler', StandardScaler())])),
                ('stacking',
                 Sta...
                                                 <catboost.core.CatBoostClassifier object at 0x78f67957fbc0>),
                                                ('lgbm',
                                                 LGBMClassifier(colsample_bytree=0.5159046186312849,
                                                                learning_rate=0.09981859646361525,
                                                                max_depth=3,
                                                                min_child_samples=86,
                                                                n_estimators=838,
                                                                num_leaves=151,
                                                                random_state=42,
                                                                subsample=0.9291146360119862,
                                                                verbose=-1))],
                                    final_estimator=LogisticRegression(),
                                    n_jobs=-1, stack_method='predict_proba'))])

In [13]:
test = pd.read_csv(f'/kaggle/input/playground-series-s6e2/test.csv')
test_id = test['id']

test_preds = stacking_model.predict_proba(test)[:, 1]

submission = pd.DataFrame({
    'id': test_id,
    'Heart Disease': test_preds
})

submission.to_csv('submission.csv', index=False)

print("Submission file saved as submission.csv")
submission.head()

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Submission file saved as submission.csv


,id,Heart Disease
0,630000,0.947532
1,630001,0.040153
2,630002,0.960255
3,630003,0.039585
4,630004,0.121420
